In [1]:
import OMPython
import os
import logging
import pandas as pd
import DyMat
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
logging.getLogger("OMPython").setLevel(logging.ERROR)
omc = OMPython.OMCSessionLocal()

# Put your thermopower pacakge paths and resulting directory
omc.sendExpression("loadFile(\"/home/wspark/projects/ThermoPower/ThermoPower/package.mo\")")
omc.sendExpression("loadFile(\"/home/wspark/projects/AmazonRunScripts/ReferenceOM/package.mo\")")
RESULT_DIR_PID = "/home/wspark/projects/OMResults/FastLoad/PID"
RESULT_DIR_SMC = "/home/wspark/projects/OMResults/FastLoad/SMC"

In [ ]:
"Simulation setting"

var_map_PID = {
    "T_helium": "sens_T_He.T",
    "T_steam": "sens_T_steam.T",
    "P_steam": "sens_P_steam.p",
    "f_grid": "sens_frequency.f",
    "V_condenser": "Condenser.condenserShell.ratio_VvVtot",
    "P_mech": "generator.Pm",
    "P_elec": "generator.Pe",
    "P_reactor": "reactorPower_Avg.P",
    "P_demand": "LoadDemand.y",
    "T_fuel_hot": "Fuel_mixture_Hot.Tmax",
    "T_fuel_avg": "Fuel_mixture_Avg.Tmax",
    "rho_der_y": "RhoDerivative.y",
    "rho_tot": "reactorPower_Avg.rho",
    "rho_Xe": "reactorPower_Avg.rho_Xe",
    "rho_ext": "reactorPower_Avg.rho_ext",
    "rho_feedback": "reactorPower_Avg.rho_feedback",
    "PID_blower_y": "PID_blower.y",
    "PID_fwpp_y": "PID_fwpp.y",
    "PID_cdpp_y": "PID_cdpp.y",
    "PID_reactivity_y": "PID_reactivity.y",
    "PID_valve_y": "PID_valve.y",
}

var_map_SMC = {
    "T_helium": "sens_T_He.T",
    "T_steam": "sens_T_steam.T",
    "P_steam": "sens_P_steam.p",
    "f_grid": "sens_frequency.f",
    "V_condenser": "Condenser.condenserShell.ratio_VvVtot",
    "P_mech": "generator.Pm",
    "P_elec": "generator.Pe",
    "P_reactor": "reactorPower_Avg.P",
    "P_demand": "LoadDemand.y",
    "T_fuel_hot": "Fuel_mixture_Hot.Tmax",
    "T_fuel_avg": "Fuel_mixture_Avg.Tmax",
    "rho_der_y": "RhoDerivative.y",
    "rho_tot": "reactorPower_Avg.rho",
    "rho_Xe": "reactorPower_Avg.rho_Xe",
    "rho_ext": "reactorPower_Avg.rho_ext",
    "rho_feedback": "reactorPower_Avg.rho_feedback",
    "SMC_blower_y": "SMC_blower.y",
    "SMC_fwpp_y": "SMC_fwpp.y",
    "SMC_cdpp_y": "SMC_cdpp.y",
    "SMC_reactivity_y": "SMC_reactivity.y",
    "SMC_valve_y": "SMC_valve.y",
}

VARS_TO_SAVE_PID = list(var_map_PID.values())
VARIABLE_FILTER_PID = "|".join(VARS_TO_SAVE_PID)
VARS_TO_SAVE_SMC = list(var_map_SMC.values())
VARIABLE_FILTER_SMC = "|".join(VARS_TO_SAVE_SMC)

STARTTIME = 0
LOADSTARTTIME = 3000
STOPTIME = LOADSTARTTIME + 3600
DT = 0.2
NUMOFINTERVAL = int((STOPTIME-STARTTIME)/DT)

NOMINAL_T_STEAM = 823
NOMINAL_P_STEAM = 14.3e6
NOMINAL_T_HELIUM = 1023
NOMINAL_F_GRID = 60
NOMINAL_P_DEMAND = 65.5e6

MODEL_PID = "ReferenceOM.TransientRuns.FastLoad_PID"
MODEL_SMC = "ReferenceOM.TransientRuns.FastLoad_SMC"
CASENAME_PID_LLAMA2 = [
    'N_1_llama2',

    'N_5_llama2',
    'N_5_hfm_energy_llama2',
    'N_5_bfm_energy_llama2',
    'N_5_hfm_std_llama2',
    'N_5_bfm_std_llama2',

    'N_25_llama2',
    ]

CASENAME_SMC_LLAMA2 = [
    'N_1_llama2',

    'N_5_llama2',
    'N_5_hfm_energy_llama2',
    'N_5_bfm_energy_llama2',
    'N_5_hfm_std_llama2',
    'N_5_bfm_std_llama2',

    'N_25_llama2',
    ]

In [ ]:
"Run PID cases"

omc.sendExpression(f'cd("{RESULT_DIR_PID}")')

for case_name in CASENAME_PID_LLAMA2:
    # Put your OM-readable DC load input
    dcFile = np.loadtxt(f"/home/wspark/projects/AmazonRunScripts/Pythons/dataCenterLoad/High_res_profile/{case_name}.txt", skiprows=2, usecols=1)
    initLoad = dcFile[0]
    height = -(1-initLoad)

    simflags = (
        # Put your OM-readable DC load input
        f'-override FastLoadDemand.fileName=\\"/home/wspark/projects/AmazonRunScripts/Pythons/dataCenterLoad/High_res_profile/{case_name}.txt\\",'
        f'preSignal.height={height}'    # This is to match case-specific initial demand
    )

    print(f"Running {case_name} in {RESULT_DIR_PID}")
    result = omc.sendExpression(f"""
    simulate(
        {MODEL_PID},
        stopTime={STOPTIME},
        numberOfIntervals={NUMOFINTERVAL},
        tolerance=1e-6,
        outputFormat="mat",
        fileNamePrefix="{case_name}",
        variableFilter="{VARIABLE_FILTER_PID}",
        simflags="{simflags}"
    )
    """)
    print("  ->", result["resultFile"])

for filename in os.listdir(RESULT_DIR_PID):
    path = os.path.join(RESULT_DIR_PID, filename)
    if os.path.isfile(path) and not filename.endswith(".mat"):
        os.remove(path)

saved_vars = omc.sendExpression(
    f'readSimulationResultVars("{result["resultFile"]}", false)'
)
for v in saved_vars:
    print(v)

In [ ]:
"Run SMC cases"

omc.sendExpression(f'cd("{RESULT_DIR_SMC}")')

for case_name in CASENAME_SMC_LLAMA2:
    # Put your OM-readable DC load input
    dcFile = np.loadtxt(f"/home/wspark/projects/AmazonRunScripts/Pythons/dataCenterLoad/High_res_profile/{case_name}.txt", skiprows=2, usecols=1)
    initLoad = dcFile[0]
    height = -(1-initLoad)

    simflags = (
        # Put your OM-readable DC load input
        f'-override FastLoadDemand.fileName=\\"/home/wspark/projects/AmazonRunScripts/Pythons/dataCenterLoad/High_res_profile/{case_name}.txt\\",'
        f'preSignal.height={height}'    # This is to match case-specific initial demand
    )

    print(f"Running {case_name} in {RESULT_DIR_SMC}")
    result = omc.sendExpression(f"""
    simulate(
        {MODEL_SMC},
        stopTime={STOPTIME},
        numberOfIntervals={NUMOFINTERVAL},
        tolerance=1e-6,
        outputFormat="mat",
        fileNamePrefix="{case_name}",
        variableFilter="{VARIABLE_FILTER_SMC}",
        simflags="{simflags}"
    )
    """)
    print("  ->", result["resultFile"])

for filename in os.listdir(RESULT_DIR_SMC):
    path = os.path.join(RESULT_DIR_SMC, filename)
    if os.path.isfile(path) and not filename.endswith(".mat"):
        os.remove(path)

saved_vars = omc.sendExpression(
    f'readSimulationResultVars("{result["resultFile"]}", false)'
)
for v in saved_vars:
    print(v)

In [ ]:
"Load PID result .mat"

data_PID_llama2 = {}
for case_name in CASENAME_PID_LLAMA2:
        mat = DyMat.DyMatFile(RESULT_DIR_PID + "/" + case_name + "_res.mat")
        time = mat.abscissa("sens_T_He.T", valuesOnly=True)
        time_analysis = LOADSTARTTIME
        idx_trim = time >= time_analysis
        time_trim = time[idx_trim]-time_analysis

        data_PID_llama2[case_name] = {}
        data_PID_llama2[case_name]["time"] = time_trim

        for name, omname in var_map_PID.items():
            data_PID_llama2[case_name][name] = mat.data(omname)[idx_trim]
        
        data_PID_llama2[case_name]["T_helium_dev"] = data_PID_llama2[case_name]["T_helium"]-NOMINAL_T_HELIUM
        data_PID_llama2[case_name]["T_steam_dev"] = data_PID_llama2[case_name]["T_steam"]-NOMINAL_T_STEAM
        data_PID_llama2[case_name]["P_steam_dev"] = (data_PID_llama2[case_name]["P_steam"]-NOMINAL_P_STEAM)/NOMINAL_P_STEAM*100
        data_PID_llama2[case_name]["rho_der_y"] = data_PID_llama2[case_name]["rho_der_y"]*1e5
        data_PID_llama2[case_name]["T_fuel_hot"] = data_PID_llama2[case_name]["T_fuel_hot"]-273.15
        data_PID_llama2[case_name]["f_grid_dev"] = (data_PID_llama2[case_name]["f_grid"]-NOMINAL_F_GRID)*1e3
        data_PID_llama2[case_name]["P_mech_dev"] = (data_PID_llama2[case_name]["P_mech"]-data_PID_llama2[case_name]["P_demand"]*NOMINAL_P_DEMAND)/NOMINAL_P_DEMAND*100
        data_PID_llama2[case_name]["P_elec_dev"] = (data_PID_llama2[case_name]["P_elec"]-data_PID_llama2[case_name]["P_demand"]*NOMINAL_P_DEMAND)/NOMINAL_P_DEMAND*100
        data_PID_llama2[case_name]["P_inertia_dev"] = data_PID_llama2[case_name]["P_elec"]-data_PID_llama2[case_name]["P_mech"]

In [ ]:
"Load SMC result .mat"

data_SMC_llama2 = {}
for case_name in CASENAME_SMC_LLAMA2:
        mat = DyMat.DyMatFile(RESULT_DIR_SMC + "/" + case_name + "_res.mat")
        time = mat.abscissa("sens_T_He.T", valuesOnly=True)
        time_analysis = LOADSTARTTIME
        idx_trim = time >= time_analysis
        time_trim = time[idx_trim]-time_analysis

        data_SMC_llama2[case_name] = {}
        data_SMC_llama2[case_name]["time"] = time_trim

        for name, omname in var_map_SMC.items():
            data_SMC_llama2[case_name][name] = mat.data(omname)[idx_trim]
            
        data_SMC_llama2[case_name]["T_helium_dev"] = data_SMC_llama2[case_name]["T_helium"]-NOMINAL_T_HELIUM
        data_SMC_llama2[case_name]["T_steam_dev"] = data_SMC_llama2[case_name]["T_steam"]-NOMINAL_T_STEAM
        data_SMC_llama2[case_name]["P_steam_dev"] = (data_SMC_llama2[case_name]["P_steam"]-NOMINAL_P_STEAM)/NOMINAL_P_STEAM*100
        data_SMC_llama2[case_name]["rho_der_y"] = data_SMC_llama2[case_name]["rho_der_y"]*1e5
        data_SMC_llama2[case_name]["T_fuel_hot"] = data_SMC_llama2[case_name]["T_fuel_hot"]-273.15
        data_SMC_llama2[case_name]["f_grid_dev"] = (data_SMC_llama2[case_name]["f_grid"]-NOMINAL_F_GRID)*1e3
        data_SMC_llama2[case_name]["P_mech_dev"] = (data_SMC_llama2[case_name]["P_mech"]-data_SMC_llama2[case_name]["P_demand"]*NOMINAL_P_DEMAND)/NOMINAL_P_DEMAND*100
        data_SMC_llama2[case_name]["P_elec_dev"] = (data_SMC_llama2[case_name]["P_elec"]-data_SMC_llama2[case_name]["P_demand"]*NOMINAL_P_DEMAND)/NOMINAL_P_DEMAND*100
        data_SMC_llama2[case_name]["P_inertia_dev"] = data_SMC_llama2[case_name]["P_elec"]-data_SMC_llama2[case_name]["P_mech"]

In [ ]:
"Generate N effect table in LaTeX form"

def get_metrics(x, absolute=True):
    x_eval = np.abs(x) if absolute else x

    return [
        np.max(x_eval),
        np.percentile(x_eval, 99),
        np.percentile(x_eval, 95),
        np.sqrt(np.mean(x**2)),
    ]

casenames = ["N_1_llama2", "N_5_llama2", "N_25_llama2"]
varnames = ["T_steam_dev", "P_steam_dev", "T_helium_dev", "P_elec_dev", "f_grid_dev", "rho_der_y", "T_fuel_hot"]
signed_vars = {"T_steam_dev", "P_steam_dev"}

rows = []
for varname in varnames:
    PI_result = []
    SMC_result = []
    use_absolute = varname not in signed_vars

    for casename in casenames:
        PI_result.append(get_metrics(data_PID_llama2[casename][varname], absolute=use_absolute))
        SMC_result.append(get_metrics(data_SMC_llama2[casename][varname], absolute=use_absolute))

    for i in range(4):
        rows.append([
            PI_result[0][i],
            PI_result[1][i],
            PI_result[2][i],
            SMC_result[0][i],
            SMC_result[1][i],
            SMC_result[2][i],
        ])


index = pd.MultiIndex.from_tuples([
    ("Steam temperature [\\degree C]", "max"),
    ("Steam temperature [\\degree C]", "p99"),
    ("Steam temperature [\\degree C]", "p95"),
    ("Steam temperature [\\degree C]", "rms"),

    ("Steam pressure  [\\%]", "max"),
    ("Steam pressure  [\\%]", "p99"),
    ("Steam pressure  [\\%]", "p95"),
    ("Steam pressure  [\\%]", "rms"),

    ("Helium temperature [\\degree C]", "max"),
    ("Helium temperature [\\degree C]", "p99"),
    ("Helium temperature [\\degree C]", "p95"),
    ("Helium temperature [\\degree C]", "rms"),

    ("Load demand tracking error [\\%]", "max"),
    ("Load demand tracking error [\\%]", "p99"),
    ("Load demand tracking error [\\%]", "p95"),
    ("Load demand tracking error [\\%]", "rms"),

    ("Grid frequency [mHz]", "max"),
    ("Grid frequency [mHz]", "p99"),
    ("Grid frequency [mHz]", "p95"),
    ("Grid frequency [mHz]", "rms"),

    ("Reactivity insertion rate [pcm/s]", "max"),
    ("Reactivity insertion rate [pcm/s]", "p99"),
    ("Reactivity insertion rate [pcm/s]", "p95"),
    ("Reactivity insertion rate [pcm/s]", "rms"),

    ("Maximum fuel temperature [\\degree C]", "max"),
    ("Maximum fuel temperature [\\degree C]", "p99"),
    ("Maximum fuel temperature [\\degree C]", "p95"),
    ("Maximum fuel temperature [\\degree C]", "rms"),
], names=[None, None])

columns = pd.MultiIndex.from_tuples([
    ("PI control", "N=1"),
    ("PI control", "N=5"),
    ("PI control", "N=25"),
    ("SMC control", "N=1"),
    ("SMC control", "N=5"),
    ("SMC control", "N=25"),
])

df = pd.DataFrame(
    rows,
    index=index,
    columns=columns,
)

styler = (df.style.format("{:.1f}"))
latex = styler.to_latex(
    caption="System responses under the workload-resolved profiles with different aggregation factors. Bold numbers indicate cases that exceed the prescribed transient operating constraints.",
    label="tab:workloadresolved_Neffect",
    position="htpb",
    position_float="centering",
    hrules=True,
    sparse_index=True,
    sparse_columns=True,
    multirow_align="c",
    multicol_align="c",
    column_format=r"l@{\hspace{8mm}}l@{\hspace{8mm}}cccccc",
    clines="skip-last;data",
)

latex = latex.replace(
    r" &  & \multicolumn{3}{c}{PI control} & \multicolumn{3}{c}{SMC control} \\",
    r"\multirow{2}{*}{Parameter} & \multirow{2}{*}{Metric} & \multicolumn{3}{c}{PI control} & \multicolumn{3}{c}{SMC control} \\"
)

latex = latex.replace(
    r" &  & N=1 & N=5 & N=25 & N=1 & N=5 & N=25 \\",
    r"\cmidrule(lr){3-5}" "\n"
    r"\cmidrule(lr){6-8}" "\n"
    r" & & N=1 & N=5 & N=25 & N=1 & N=5 & N=25 \\"
)

print(latex)

\begin{table}[htpb]
\centering
\caption{System responses under the workload-resolved profiles with different aggregation factors. Bold numbers indicate cases that exceed the prescribed transient operating constraints.}
\label{tab:workloadresolved_Neffect}
\begin{tabular}{l@{\hspace{8mm}}l@{\hspace{8mm}}cccccc}
\toprule
\multirow{2}{*}{Parameter} & \multirow{2}{*}{Metric} & \multicolumn{3}{c}{PI control} & \multicolumn{3}{c}{SMC control} \\
\cmidrule(lr){3-5}
\cmidrule(lr){6-8}
 & & N=1 & N=5 & N=25 & N=1 & N=5 & N=25 \\
\midrule
\multirow[c]{4}{*}{Steam temperature [\degree C]} & max & 26.5 & 17.8 & 8.0 & 27.2 & 18.3 & 7.6 \\
 & p99 & 20.3 & 9.9 & 5.6 & 20.1 & 9.3 & 5.2 \\
 & p95 & 16.1 & 7.2 & 3.7 & 15.3 & 6.2 & 3.1 \\
 & rms & 10.1 & 4.3 & 2.3 & 9.0 & 3.6 & 1.8 \\
\cline{1-8}
\multirow[c]{4}{*}{Steam pressure  [\%]} & max & 14.0 & 8.5 & 3.4 & 21.7 & 9.6 & 3.1 \\
 & p99 & 9.2 & 4.2 & 2.2 & 14.4 & 4.1 & 1.7 \\
 & p95 & 6.9 & 3.0 & 1.5 & 8.3 & 2.7 & 1.0 \\
 & rms & 3.9 & 1.7 & 0.9 & 4.3

In [ ]:
"Generate band effect table in LaTeX form"

def get_metrics(x, absolute=True):
    x_eval = np.abs(x) if absolute else x

    return [
        np.max(x_eval),
        np.percentile(x_eval, 99),
        np.percentile(x_eval, 95),
        np.sqrt(np.mean(x**2)),
    ]

casenames = ["N_5_llama2", "N_5_hfm_energy_llama2", "N_5_bfm_energy_llama2", "N_5_hfm_std_llama2", "N_5_bfm_std_llama2"]
varnames = ["T_steam_dev", "P_steam_dev", "T_helium_dev", "P_elec_dev", "f_grid_dev", "rho_der_y", "T_fuel_hot"]
signed_vars = {"T_steam_dev", "P_steam_dev"}

rows = []

for varname in varnames:
    PID_result = []
    SMC_result = []
    use_absolute = varname not in signed_vars

    for casename in casenames:
        PID_result.append(
            get_metrics(data_PID_llama2[casename][varname], absolute=use_absolute)
        )
        SMC_result.append(
            get_metrics(data_SMC_llama2[casename][varname], absolute=use_absolute)
        )

    for i in range(4):
        rows.append([
            f"{PID_result[0][i]:.1f} / {SMC_result[0][i]:.1f}",
            f"{PID_result[1][i]:.1f} / {SMC_result[1][i]:.1f}",
            f"{PID_result[2][i]:.1f} / {SMC_result[2][i]:.1f}",
            f"{PID_result[3][i]:.1f} / {SMC_result[3][i]:.1f}",
            f"{PID_result[4][i]:.1f} / {SMC_result[4][i]:.1f}",
        ])

index = pd.MultiIndex.from_tuples([
    (r"\makecell[l]{Steam \\temperature [\degree C]}", "max"),
    (r"\makecell[l]{Steam \\temperature [\degree C]}", "p99"),
    (r"\makecell[l]{Steam \\temperature [\degree C]}", "p95"),
    (r"\makecell[l]{Steam \\temperature [\degree C]}", "rms"),

    (r"\makecell[l]{Steam \\pressure [\%]}", "max"),
    (r"\makecell[l]{Steam \\pressure [\%]}", "p99"),
    (r"\makecell[l]{Steam \\pressure [\%]}", "p95"),
    (r"\makecell[l]{Steam \\pressure [\%]}", "rms"),

    (r"\makecell[l]{Helium \\temperature [\degree C]}", "max"),
    (r"\makecell[l]{Helium \\temperature [\degree C]}", "p99"),
    (r"\makecell[l]{Helium \\temperature [\degree C]}", "p95"),
    (r"\makecell[l]{Helium \\temperature [\degree C]}", "rms"),

    (r"\makecell[l]{Load demand\\tracking error [\%]}", "max"),
    (r"\makecell[l]{Load demand\\tracking error [\%]}", "p99"),
    (r"\makecell[l]{Load demand\\tracking error [\%]}", "p95"),
    (r"\makecell[l]{Load demand\\tracking error [\%]}", "rms"),

    (r"\makecell[l]{Grid \\frequency [mHz]}", "max"),
    (r"\makecell[l]{Grid \\frequency [mHz]}", "p99"),
    (r"\makecell[l]{Grid \\frequency [mHz]}", "p95"),
    (r"\makecell[l]{Grid \\frequency [mHz]}", "rms"),

    (r"\makecell[l]{Reactivity insertion\\rate [pcm/s]}", "max"),
    (r"\makecell[l]{Reactivity insertion\\rate [pcm/s]}", "p99"),
    (r"\makecell[l]{Reactivity insertion\\rate [pcm/s]}", "p95"),
    (r"\makecell[l]{Reactivity insertion\\rate [pcm/s]}", "rms"),

    (r"\makecell[l]{Maximum fuel\\temperature [\degree C]}", "max"),
    (r"\makecell[l]{Maximum fuel\\temperature [\degree C]}", "p99"),
    (r"\makecell[l]{Maximum fuel\\temperature [\degree C]}", "p95"),
    (r"\makecell[l]{Maximum fuel\\temperature [\degree C]}", "rms"),
], names=[None, None])

columns = pd.MultiIndex.from_tuples([
    ("Signal case", "Original"),
    ("Signal case", "HFM-energy"),
    ("Signal case", "IFM-energy"),
    ("Signal case", "HFM-std"),
    ("Signal case", "IFM-std"),
])

df = pd.DataFrame(
    rows,
    index=index,
    columns=columns,
)

styler = df.style
latex = styler.to_latex(
    caption="Effects of frequency-band removal on system responses. Values are shown as PI/SMC cases. Bold values highlight the representative results discussed in the main text.",
    label="tab:workloadresolved_freqmv",
    position="htpb",
    position_float="centering",
    hrules=True,
    sparse_index=True,
    sparse_columns=True,
    multirow_align="c",
    multicol_align="c",
    column_format=r"l@{\hspace{3mm}}l@{\hspace{1mm}}ccccc",
    clines="skip-last;data",
)

latex = latex.replace(
    r" &  & \multicolumn{5}{c}{Signal case} \\",
    r"\multirow{2}{*}{Parameter} & \multirow{2}{*}{Metric} & \multicolumn{5}{c}{Signal case} \\"
)

latex = latex.replace(
    r" &  & Original & HFM-energy & IFM-energy & HFM-std & IFM-std \\",
    r"\cmidrule(lr){3-7}" "\n"
    r" & & Original & HFM-energy & IFM-energy & HFM-std & IFM-std \\"
)

print(latex)

\begin{table}[htpb]
\centering
\caption{Effects of frequency-band removal on system responses. Values are shown as PI/SMC cases. Bold values highlight the representative results discussed in the main text.}
\label{tab:workloadresolved_freqmv}
\begin{tabular}{l@{\hspace{3mm}}l@{\hspace{1mm}}ccccc}
\toprule
\multirow{2}{*}{Parameter} & \multirow{2}{*}{Metric} & \multicolumn{5}{c}{Signal case} \\
\cmidrule(lr){3-7}
 & & Original & HFM-energy & IFM-energy & HFM-std & IFM-std \\
\midrule
\multirow[c]{4}{*}{\makecell[l]{Steam \\temperature [\degree C]}} & max & 17.8 / 18.3 & 15.1 / 14.9 & 9.6 / 7.9 & 24.5 / 25.8 & 14.3 / 12.4 \\
 & p99 & 9.9 / 9.3 & 7.6 / 6.7 & 5.1 / 4.3 & 11.4 / 10.4 & 7.4 / 6.5 \\
 & p95 & 7.2 / 6.2 & 5.5 / 4.5 & 3.6 / 2.9 & 8.3 / 6.8 & 5.2 / 4.4 \\
 & rms & 4.3 / 3.6 & 3.4 / 2.6 & 2.1 / 1.6 & 5.2 / 4.1 & 3.1 / 2.5 \\
\cline{1-7}
\multirow[c]{4}{*}{\makecell[l]{Steam \\pressure [\%]}} & max & 8.5 / 9.6 & 7.4 / 8.3 & 3.6 / 3.6 & 12.1 / 14.2 & 5.2 / 5.5 \\
 & p99 & 4.2 / 4.1